In [21]:
pip install transformers datasets scikit-learn seaborn

In [22]:
import torch
from transformers import RobertaModel, RobertaTokenizer, RobertaForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import copy
import random

In [48]:
# ─── Model configuration ───────────────────────────────────────────────────
MODEL_NAME = 'roberta-large'
HIDDEN_SIZE = 1024   # roberta-base: 768, roberta-large: 1024
NUM_LAYERS = 24      # roberta-base: 12,  roberta-large: 24
RANK = 4
LORA_ALPHA = 1.0
BATCH_SIZE = 128
NUM_EPOCHS = 2
LR = 3e-4
NUM_CLIENTS = 4
SEED = 2026
MAX_SAMPLES_PER_CLIENT = None


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)
  device = "cuda"
else:
  device ="cpu"


print(f"Device : {device}")

Device : cuda


In [49]:
# Load RoBERTa tokenizer and model from HuggingFace
# The tokenizer converts raw text to token IDs
# The model weights are frozen - only LoRA parameters will be trained
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
#model = RobertaModel.from_pretrained(MODEL_NAME)

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

# Freeze all pretrained model parameters.
# We only train the manually defined LoRA parameters.
for param in model.parameters():
    param.requires_grad = False

# Keep the classifier frozen too, to make the comparison focus on LoRA.
# This is closer to the paper, where the classification head is frozen after initialization.
for param in model.classifier.parameters():
    param.requires_grad = False

# Print model config to verify architecture
print(model.config)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [50]:
# Load SST-2 dataset from the GLUE benchmark
# SST-2 is a binary sentiment classification task (positive/negative)
dataset_sst2 = load_dataset('glue', 'sst2')
print(dataset_sst2)
print(dataset_sst2['train'][0])

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [51]:
# Load QNLI dataset from the GLUE benchmark
# QNLI is a question-answer inference task: given a question and a sentence,
# determine whether the sentence contains the answer to the question
dataset_qnli = load_dataset('glue', 'qnli')
print(dataset_qnli)
print(dataset_qnli['train'][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})
{'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}


In [52]:
def tokenize_sst2(examples):
    """
    Tokenize SST-2 examples.
    SST-2 has a single text input (sentence).
    """
    return tokenizer(
        examples['sentence'],
        truncation=True,      # Truncate sequences longer than max_length
        max_length=128,       # Maximum sequence length (paper uses 128)
        padding='max_length'  # Pad shorter sequences to max_length
    )

def tokenize_qnli(examples):
    """
    Tokenize QNLI examples.
    QNLI has two text inputs (question + sentence) that are concatenated
    by the tokenizer with a separator token.
    """
    return tokenizer(
        examples['question'],
        examples['sentence'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

In [53]:
# Apply tokenization to both datasets using batched processing for efficiency
# This adds 'input_ids' and 'attention_mask' columns to each dataset
dataset_sst2 = dataset_sst2.map(tokenize_sst2, batched=True)
dataset_qnli = dataset_qnli.map(tokenize_qnli, batched=True)

# Verify that tokenization columns were added correctly
print(dataset_sst2['train'].column_names)

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

['sentence', 'label', 'idx', 'input_ids', 'attention_mask']


In [54]:
# Remove unnecessary columns and keep only: input_ids, attention_mask, label
# Then convert datasets to PyTorch tensor format
dataset_sst2 = dataset_sst2.remove_columns(['idx', 'sentence'])
dataset_sst2 = dataset_sst2.with_format('torch')

dataset_qnli = dataset_qnli.remove_columns(['idx', 'question', 'sentence'])
dataset_qnli = dataset_qnli.with_format('torch')

# Verify that only the necessary columns remain
print(dataset_sst2['train'].column_names)
print(dataset_qnli['train'].column_names)

['label', 'input_ids', 'attention_mask']
['label', 'input_ids', 'attention_mask']


In [55]:
def partition_dataset(dataset, num_clients, max_samples_per_client=None):
    """
    Randomly partition a dataset into equal parts for each client.
    Each client receives a non-overlapping subset of the data.

    Args:
        dataset: HuggingFace dataset to partition
        num_clients: number of clients to split the data among

    Returns:
        list of dataset subsets, one per client
    """
    n = len(dataset)

    # Randomly shuffle indices to ensure each client gets a random subset
    indices = np.random.permutation(n)
    if max_samples_per_client is not None:
      total_needed = num_clients * max_samples_per_client
      indices = indices[:total_needed]
      size = max_samples_per_client
    else:
      # Size of each client's partition
      size = n // num_clients

    partitions = []

    for i in range(num_clients):
        # Compute start and end indices for client i
        start = i * size
        end = (i + 1) * size
        partitions.append(dataset.select(indices[start:end]))

    return partitions

In [56]:
# Partition SST-2 among clients 0 and 1, QNLI among clients 2 and 3
# This simulates the heterogeneous federated learning setting from the paper:
# clients 0-1 share the same task (SST-2), clients 2-3 share a different task (QNLI)
sst2_partitions = partition_dataset(dataset_sst2['train'], num_clients=2, max_samples_per_client=MAX_SAMPLES_PER_CLIENT)
qnli_partitions = partition_dataset(dataset_qnli['train'], num_clients=2, max_samples_per_client=MAX_SAMPLES_PER_CLIENT)

# Map each client ID to its local dataset
client_datasets = {
    0: sst2_partitions[0],  # SST-2
    1: sst2_partitions[1],  # SST-2
    2: qnli_partitions[0],  # QNLI
    3: qnli_partitions[1],  # QNLI
}

# Verify partition sizes
for cid, ds in client_datasets.items():
    print(f"Client {cid} : {len(ds)} examples")

Client 0 : 33674 examples
Client 1 : 33674 examples
Client 2 : 52371 examples
Client 3 : 52371 examples


In [57]:
# Create a DataLoader for each client
# batch_size=32 (paper uses 128, reduced here for Colab T4 memory constraints)
# shuffle=True to ensure random ordering of samples during training
client_loaders = {}
for cid, ds in client_datasets.items():
    client_loaders[cid] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

# Verify number of batches per client
print(f"Number of batches per client:")
for cid, loader in client_loaders.items():
    print(f"  Client {cid} : {len(loader)} batches")

Number of batches per client:
  Client 0 : 264 batches
  Client 1 : 264 batches
  Client 2 : 410 batches
  Client 3 : 410 batches


In [58]:
import math

def initialize_lora_params(rank=RANK):
    """
    Initialize LoRA parameters (A and B matrices) for all query and value
    projections across all layers of RoBERTa.

    Returns a dictionary: layer_name -> {'A': tensor, 'B': tensor}
    """
    lora_params = {}

    for layer_idx in range(NUM_LAYERS):
        for proj in ['query', 'value']:
            key = f'layer_{layer_idx}_{proj}'

            # A : shape (rank, HIDDEN_SIZE), initialized with kaiming uniform
            A = torch.empty(RANK, HIDDEN_SIZE)
            torch.nn.init.kaiming_uniform_(A, a=math.sqrt(5))

            # B : shape (HIDDEN_SIZE, rank), initialized to zero so that
            # the LoRA delta B@A = 0 at the start of training,
            # leaving the pretrained model unchanged (paper Section 3.1)
            B = torch.zeros(HIDDEN_SIZE, RANK)

            lora_params[key] = {
                'A': A.to(device),
                'B': B.to(device)
            }

    return lora_params

In [59]:
# Verify LoRA parameter initialization for one client
# Expected: NUM_LAYERS * 2 layers (query and value projections)
# A shape: (RANK, HIDDEN_SIZE)
# B shape: (HIDDEN_SIZE, RANK)
lora_params_client0 = initialize_lora_params(rank=RANK)
print(f"Number of LoRA layers: {len(lora_params_client0)}")
print(f"Shape of A: {lora_params_client0['layer_0_query']['A'].shape}")
print(f"Shape of B: {lora_params_client0['layer_0_query']['B'].shape}")

Number of LoRA layers: 48
Shape of A: torch.Size([4, 1024])
Shape of B: torch.Size([1024, 4])


In [60]:
# Initialize independent LoRA parameters for each client
# Each client starts with the same initialization but will diverge during local training
# reflecting their different task distributions (SST-2 vs QNLI)
initial_lora_params = initialize_lora_params(rank=RANK)

client_lora_params = {
    cid: copy.deepcopy(initial_lora_params)
    for cid in range(NUM_CLIENTS)
}

In [61]:
def lora_forward(model, input_ids, attention_mask, lora_params, rank=RANK, lora_alpha=LORA_ALPHA):
    """
    Forward pass through RobertaForSequenceClassification with LoRA deltas
    applied to the query and value projections.

    The model already includes the classification head, so this function
    directly returns the classification logits.
    """
    scaling = lora_alpha / rank
    hooks = []

    def make_hook(key):
        def hook(module, input, output):
            # input[0] shape: (batch, seq_len, hidden_size)
            A = lora_params[key]['A']
            B = lora_params[key]['B']

            # LoRA delta: x @ A.T @ B.T
            lora_delta = input[0] @ A.T @ B.T * scaling

            return output + lora_delta

        return hook

    # Register LoRA hooks on query and value projections
    for layer_idx in range(NUM_LAYERS):
        for proj in ['query', 'value']:
            key = f'layer_{layer_idx}_{proj}'

            # With RobertaForSequenceClassification, the backbone is model.roberta
            layer = model.roberta.encoder.layer[layer_idx].attention.self

            hook = getattr(layer, proj).register_forward_hook(make_hook(key))
            hooks.append(hook)

    # Forward pass through the full sequence classification model
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    # Remove hooks after the forward pass
    for hook in hooks:
        hook.remove()

    # Return logits directly
    return outputs.logits

In [62]:
# Test lora_forward with a small batch
# Expected output shape: (batch_size, num_labels)
model.eval()

batch = next(iter(client_loaders[0]))
input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)

with torch.no_grad():
    logits = lora_forward(model, input_ids, attention_mask, client_lora_params[0])

print(f"Logits shape: {logits.shape}")

Logits shape: torch.Size([128, 2])


In [63]:
import torch.nn as nn

class ClassificationHead(nn.Module):
    """
    Simple classification head on top of RoBERTa.
    Takes the [CLS] token representation and predicts the class.

    The [CLS] token (index 0) aggregates the full sequence representation
    and is standard practice for classification tasks with BERT-based models.
    """
    def __init__(self, hidden_size=HIDDEN_SIZE, num_classes=2):
        super().__init__()
        # Linear layer mapping from HIDDEN_SIZE to number of classes (2)
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, roberta_output):
        # Extract [CLS] token (index 0) from all sequences in the batch
        # Shape: (batch_size, HIDDEN_SIZE)
        cls_token = roberta_output.last_hidden_state[:, 0, :]
        # Shape: (batch_size, num_classes)
        return self.linear(cls_token)

# Each client has its own classification head trained on its local data
client_heads = {
    cid: ClassificationHead().to(device) for cid in range(NUM_CLIENTS)
}

print(f"Number of classification heads: {len(client_heads)}")

Number of classification heads: 4


In [64]:
def train_client(model, lora_params, loader, num_epochs=NUM_EPOCHS, lr=LR):
    """
    Train one client locally for num_epochs epochs.

    Only the manual LoRA parameters A and B are updated.
    The RoBERTa backbone and its classification head stay frozen.
    """
    trainable_params = []

    # Convert A and B tensors to nn.Parameters so they can be optimized
    for key in lora_params:
        if not isinstance(lora_params[key]['A'], nn.Parameter):
            lora_params[key]['A'] = nn.Parameter(lora_params[key]['A'])
        if not isinstance(lora_params[key]['B'], nn.Parameter):
            lora_params[key]['B'] = nn.Parameter(lora_params[key]['B'])

        trainable_params.append(lora_params[key]['A'])
        trainable_params.append(lora_params[key]['B'])

    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    criterion = nn.CrossEntropyLoss()
    loss_history = []

    model.train()

    for epoch in range(num_epochs):
        total_loss = 0.0

        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()

            # Forward pass directly returns classification logits
            logits = lora_forward(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                lora_params=lora_params
            )

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        loss_history.append(avg_loss)

        print(f"  Epoch {epoch + 1}/{num_epochs} | Loss: {avg_loss:.4f}")

    return lora_params, loss_history

In [ ]:
# Initialization phase:
# Each client independently trains its local LoRA for E=2 epochs.
# Clients 0-1 use SST-2, clients 2-3 use QNLI.
task_names = {
    0: 'SST-2',
    1: 'SST-2',
    2: 'QNLI',
    3: 'QNLI'
}

for cid in range(NUM_CLIENTS):
    print(f"Training client {cid} ({task_names[cid]})...")

    client_lora_params[cid], _ = train_client(
        model=model,
        lora_params=client_lora_params[cid],
        loader=client_loaders[cid],
        num_epochs=NUM_EPOCHS,
        lr=LR
    )

Training client 0 (SST-2)...


In [ ]:
def extract_layerwise_matrices(client_lora_params, matrix_type="B"):
    matrices = {}

    for cid, lora_params in client_lora_params.items():
        matrices[cid] = []

        for layer_idx in range(NUM_LAYERS):
            layer_parts = []

            for proj in ["query", "value"]:
                key = f"layer_{layer_idx}_{proj}"

                A = lora_params[key]["A"].detach().cpu()
                B = lora_params[key]["B"].detach().cpu()

                if matrix_type == "A":
                    layer_parts.append(A.flatten())
                elif matrix_type == "B":
                    layer_parts.append(B.flatten())
                elif matrix_type == "BA":
                    layer_parts.append((B @ A).flatten())

            matrices[cid].append(torch.cat(layer_parts))

    return matrices

In [ ]:
a_matrices = extract_layerwise_matrices(client_lora_params, "A")
b_matrices = extract_layerwise_matrices(client_lora_params, "B")
ba_matrices = extract_layerwise_matrices(client_lora_params, "BA")

In [ ]:
def compute_cosine_similarity(matrices, client_i, client_j):
    similarities = []

    for layer_idx in range(len(matrices[client_i])):
        x = matrices[client_i][layer_idx]
        y = matrices[client_j][layer_idx]

        sim = F.cosine_similarity(x.unsqueeze(0), y.unsqueeze(0))
        similarities.append(sim.item())

    return similarities

In [ ]:
sim_0_1 = compute_cosine_similarity(b_matrices, 0, 1)
sim_0_2 = compute_cosine_similarity(b_matrices, 0, 2)
sim_2_3 = compute_cosine_similarity(b_matrices, 2, 3)

print(f"Avg similarity client 0 vs 1 (same task SST-2): {np.mean(sim_0_1):.4f}")
print(f"Avg similarity client 0 vs 2 (different task):  {np.mean(sim_0_2):.4f}")
print(f"Avg similarity client 2 vs 3 (same task QNLI):  {np.mean(sim_2_3):.4f}")

In [ ]:
pairs = [
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 2),
    (1, 3),
    (2, 3),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

subtitles = ['(a)', '(b)', '(c)']
data = [
    (a_matrices,  'LoRA A matrices'),
    (b_matrices,  'LoRA B matrices'),
    (ba_matrices, 'LoRA BA matrices'),
]

for ax, (matrices, title), subtitle in zip(axes, data, subtitles):
    for (ci, cj) in pairs:
        sims = compute_cosine_similarity(matrices, ci, cj)
        avg = np.mean(sims)
        ax.plot(sims, label=f'Client {ci} vs {cj} (avg: {avg:.4f})')

    ax.set_xlabel('Layer')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title(f'{subtitle} {title}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.2, 1.05)

plt.suptitle(
    'Cosine similarity among clients using different LoRA matrices',
    fontsize=12,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('figure3.png', dpi=150)
plt.show()

In [ ]:
for cid in range(NUM_CLIENTS):
    b = client_lora_params[cid]['layer_0_query']['B']
    print(f"Client {cid} | B mean: {b.mean().item():.6f} | B std: {b.std().item():.6f}")

In [ ]:
# Check gradients on B matrices
for cid in range(NUM_CLIENTS):
    b = client_lora_params[cid]['layer_0_query']['B']
    print(f"Client {cid} | B grad: {b.grad}")